# 03 SQL 적재

전처리한 세 표를 PostgreSQL 데이터베이스 `cafe_location_project`에 적재하고 기본 키를 건다. 접속 정보는 프로젝트 루트의 `.env`에 있다(코드에 비밀번호를 적지 않음).

이 노트북을 다시 실행하면 표가 새로 만들어지므로, 기본 키도 이 노트북 안에서 `sql/01_기본키.sql`을 실행해 다시 건다.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

load_dotenv("../.env", override=True)
engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],
    host=os.environ["PGHOST"],
    port=int(os.environ["PGPORT"]),
    database=os.environ["PGDATABASE"],
))
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database()")).scalar())

cafe_location_project


In [2]:
OUT = Path("../data/processed")
코드_열 = {"기준_년분기_코드": str, "상권_코드": str, "자치구_코드": str, "행정동_코드": str}

for 표 in ["커피_매출", "커피_점포", "영역"]:
    df = pd.read_csv(OUT / f"{표}.csv", dtype=코드_열)
    df.to_sql(표, engine, if_exists="replace", index=False, chunksize=5000)
    with engine.connect() as conn:
        적재 = conn.execute(text(f'SELECT COUNT(*) FROM "{표}"')).scalar()
    print(표, "| 파일", len(df), "| 적재", 적재, "| 일치", len(df) == 적재)

커피_매출 | 파일 23700 | 적재 23700 | 일치 True


커피_점포 | 파일 33263 | 적재 33263 | 일치 True
영역 | 파일 1650 | 적재 1650 | 일치 True


In [3]:
# 기본 키
with engine.begin() as conn:
    for 문장 in Path("../sql/01_기본키.sql").read_text(encoding="utf-8").split(";"):
        if 문장.strip():
            conn.execute(text(문장))

pd.read_sql("""
SELECT tc.table_name AS 표, string_agg(kcu.column_name, ', ' ORDER BY kcu.ordinal_position) AS 기본_키
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu ON tc.constraint_name = kcu.constraint_name
WHERE tc.constraint_type = 'PRIMARY KEY' AND tc.table_schema = 'public'
GROUP BY tc.table_name
""", engine)

,표,기본_키
0,영역,상권_코드
1,커피_매출,"기준_년분기_코드, 상권_코드"
2,커피_점포,"기준_년분기_코드, 상권_코드"
